In [ ]:
# Methods to flag anomalies - static thresholds, z-score, IQR

In [2]:
# Static threshold
import numpy as np
rng = np.random.default_rng(seed=42)
cpu = np.clip(rng.normal(60, 5, 60), 0, 100)
cpu[[10, 25, 45]] = [92, 88, 95]  #inject 3 known spikes
threshold = 80

alerts = np.where(cpu > threshold)[0]
print(f"static threshold: {threshold}%")
print(f"alert indices: {alerts} ")
print(f"alert values:  {cpu[alerts].round(1)}")

# Problem — if a server normally runs at 75%, threshold of 80 creates constant noise. 
# If another server normally runs at 50%, a reading of 79% might be genuinely abnormal but gets missed. 
# Static thresholds do not adapt to each server's behaviour.

static threshold: 80%
alert indices: [10 25 45] 
alert values:  [92. 88. 95.]


In [6]:
# Z-Score dynamic method
# Flags readings that are unusally far from the servers own mean. Adapts to each server automatically.

import numpy as np
rng = np.random.default_rng(seed=42)
cpu = np.clip(rng.normal(60, 5, 60), 0, 100)
cpu[[10, 25, 45]] = [92, 88, 95]
mean = np.mean(cpu)
std = np.std(cpu)

# Z-score - how many std deviations from mean
z_scores = (cpu - mean) / std

# Flag anything beyond 2 standard deviations
z_threshold = 2.0
alerts = np.where(np.abs(z_scores) > z_threshold)[0]

print(f"mean: {mean:.1f} std: {std:.1f}")
print(f"z threshold: {z_threshold} ")
print(f"alert indices: {alerts} ")
print(f"alert values: {cpu[alerts].round(1)}")
print(f"z scores: {z_scores[alerts].round(2)}")

# The threshold moves with the data. A server that normally runs at 75% gets a higher threshold automatically. 
# np.abs catches both high and low anomalies — unusually low CPU can also indicate a problem like a crashed process.

mean: 61.8 std: 7.9
z threshold: 2.0 
alert indices: [10 25 45] 
alert values: [92. 88. 95.]
z scores: [3.83 3.32 4.21]


In [7]:
 #IQR Method - Inter quartile range
# IQR = the range of the “middle 50%” of your data

rng = np.random.default_rng(seed=42)
cpu = np.clip(rng.normal(60, 5, 60), 0, 100)
cpu[[10, 25, 45]] = [92, 88, 95]

q25 = np.percentile(cpu, 25)
q75 = np.percentile(cpu, 75)
iqr = q75 - q25

#standard IQR rule - 1.5 x IQR beyond the qualities
lower = q25 - 1.5 *iqr
upper = q75 - 1.5 *iqr

alerts = np.where((cpu>lower) | (cpu > upper)) [0]

print(f"q25: {q25:.1f}  q75: {q75:.1f} iqr: {iqr:.1f}")
print(f"lower bound: {lower:.1f}  upper bound: {upper:1f}")
print(f"alert indices: {alerts} ")
print(f"alert values: {cpu[alerts].round(1)}")

# IQR ignores the mean entirely. 
# It looks at where the middle 50% of data sits and flags anything too far outside that range. 
# Works well when your data is not normally distributed.

q25: 57.6  q75: 63.5 iqr: 5.9
lower bound: 48.8  upper bound: 54.656202
alert indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59] 
alert values: [61.5 54.8 63.8 64.7 50.2 53.5 60.6 58.4 59.9 55.7 92.  63.9 60.3 65.6
 62.3 55.7 61.8 55.2 64.4 59.8 59.1 56.6 66.1 59.2 57.9 88.  62.7 61.8
 62.1 62.2 70.7 58.  57.4 55.9 63.1 65.6 59.4 55.8 55.9 63.3 63.7 62.7
 56.7 61.2 60.6 95.  64.4 61.1 63.4 60.3 61.4 63.2 52.7 58.4 57.6 56.8
 58.6 67.5 55.7 64.8]


In [20]:
# comparing all three above methods
import numpy as np
rng = np.random.default_rng(seed =42)
cpu = np.clip(rng.normal(60, 5, 60), 0, 100)
cpu[[10,25,45]] = [92, 88, 95]

# static
static_alerts = set(np.where(cpu > 80)[0])

#z-score
z = (cpu - cpu.mean()) / cpu.std()
zscore_alerts = set(np.where(np.abs(z) > 2) [0])

#IQR
q25, q75 = np.percentile(cpu, [25,75])
iqr = q75 - q25
iqr_alerts = set(np.where((cpu < q25 - 1.5*iqr) | (cpu > q75 + 1.5*iqr))[0])

print(f" static alerts: {sorted(static_alerts)}")
print(f" zscore alerts: {sorted(zscore_alerts)}")
print(f" iqr alerts: {sorted(iqr_alerts)}")

# consensus - flagged by at least 2 methods, Consensus = agreement between multiple methods
# Flag anomaly only if at least 2 methods agree
# & → finds common elements (agreement)
# | → combines all agreements
# \ - backslash - “This line is not finished — continue on the next line”
consensus = static_alerts & zscore_alerts | \
            static_alerts & iqr_alerts | \
            zscore_alerts & iqr_alerts

print(f"consensus :   {sorted(consensus)}")

 static alerts: [np.int64(10), np.int64(25), np.int64(45)]
 zscore alerts: [np.int64(10), np.int64(25), np.int64(45)]
 iqr alerts: [np.int64(10), np.int64(25), np.int64(45)]
consensus :   [np.int64(10), np.int64(25), np.int64(45)]


In [23]:
# Putting it together - Incident Report for a dataset.
rng = np.random.default_rng(seed=42)
n=60
cpu = np.clip(rng.normal(60, 5, n), 0, 100)
latency = np.clip(rng.normal(15, 3, n), 0, 500)

#Inject correlated incident at minute 20-22

incident = np.arange(20, 23)
cpu[incident] = [91, 94, 88]
latency[incident] = [95, 110, 88]

# Z-score on both metrics

cpu_z = (cpu - cpu.mean()) / cpu.std()
latency_z = (latency - latency.mean() ) / latency. std()

# Flag minutes where Both metrics are anomalous

both_anomalous = np.where(np.abs(cpu_z) > 2 & (np.abs(latency_z) > 2))[0]
print(f"minutes anomalous in both metrics: {both_anomalous} ")
print(f"cpu at those minutes: {cpu[both_anomalous].round(1)}")
print(f"latency at those minutes: {latency[both_anomalous].round(1) }")

minutes anomalous in both metrics: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49 50 51 52 53 54 55 56 57 58 59] 
cpu at those minutes: [61.5 54.8 63.8 64.7 50.2 53.5 60.6 58.4 59.9 55.7 64.4 63.9 60.3 65.6
 62.3 55.7 61.8 55.2 64.4 59.8 91.  94.  88.  59.2 57.9 58.2 62.7 61.8
 62.1 62.2 70.7 58.  57.4 55.9 63.1 65.6 59.4 55.8 55.9 63.3 63.7 62.7
 56.7 61.2 60.6 61.1 64.4 61.1 63.4 60.3 61.4 63.2 52.7 58.4 57.6 56.8
 58.6 67.5 55.7 64.8]
latency at those minutes: [ 10.   14.   15.5  16.8  17.1  17.4  14.   13.6  17.6  14.4  11.2  11.6
  12.2  16.5  15.4  17.1  13.7  15.5  16.9  14.1  95.  110.   88.   13.9
  11.4  16.5  13.6  15.   16.4  16.3  17.   14.7  13.7  14.8   9.9  10.7
  11.   12.   16.2  12.3  13.9  18.9  13.9  17.2  12.2  14.4  12.1  14.
  17.5   9.8  16.3  15.7  13.2  10.7  15.2  13.4  15.7  15.1  19.8  14.3]


In [34]:
# Dataset
import numpy as np
rng = np.random.default_rng(seed = 77)
n = 120
cpu = np.clip(rng.normal(65, 6, n), 0, 100)
latency = np.clip(rng.normal(20, 4, n), 0, 500)
memory = np.clip(rng.normal(70, 8, n), 0, 100)

#Inject known anomalies
cpu[[30, 60, 90]] = [93, 91, 96]
latency [[30, 60, 90]] = [110, 95, 130]
memory[90] = 97

# 1. Static threshold - flag cpu above 85

static_alerts = set(np.where(cpu > 85)[0])
print("static alerts", static_alerts)

# 2. Apply z-score - flag cpu more than 2 std from mean

cpu_z = (cpu - cpu.mean()) / cpu.std()
zscore_alerts = set(np.where(np.abs(cpu_z) > 2) [0])
print(f"zscore alerts: {zscore_alerts} ")

# 3. Apply IQR method to cpu

q25, q75 = np.percentile(cpu, [25,75])
iqr = q75 - q25
lower = q25 - 1.5 *iqr
upper = q75 + 1.5 *iqr
iqr_alerts = set(np.where((cpu < q25 - 1.5*iqr) | (cpu > q75 + 1.5*iqr))[0])
print(f"iqr alerts: {iqr_alerts}")

# 4. Find consensus - indices flagged by all 3 methods

consensus = static_alerts & zscore_alerts | \
            static_alerts & iqr_alerts | \
            zscore_alerts & iqr_alerts

print(f"consensus : {consensus}" )

# 5. Find minutes where Both cpu and latency are anomalous, use z-score > 2 for both

latency_z = (latency - latency.mean() ) / latency. std()
both_anomalous = np.where( (np.abs(cpu_z) > 2) & (np.abs(latency_z) > 2))[0]
print(f"both anomalous: {both_anomalous}" )

# 6. Print a simple incident report for consensus alerts 

for idx in consensus:
    print(f" minute {idx:3d} - cpu: {cpu[idx]:.1f}% "
          f"latency: {latency[idx]:.1f} ms "
          f"memory: {memory[idx]:.1f}%")

static alerts {np.int64(90), np.int64(60), np.int64(30), np.int64(63)}
zscore alerts: {np.int64(2), np.int64(108), np.int64(90), np.int64(60), np.int64(30), np.int64(63)} 
iqr alerts: {np.int64(90), np.int64(60), np.int64(30), np.int64(63)}
consensus : {np.int64(90), np.int64(60), np.int64(30), np.int64(63)}
both anomalous: [30 60 90]
 minute  90 - cpu: 96.0% latency: 130.0 ms memory: 97.0%
 minute  60 - cpu: 91.0% latency: 95.0 ms memory: 63.5%
 minute  30 - cpu: 93.0% latency: 110.0 ms memory: 80.3%
 minute  63 - cpu: 87.0% latency: 20.9 ms memory: 89.1%
